# AgriCL-Net, step 7: results report

Collects the results of every finished run in `MyDrive/Colab Notebooks/agricl-net/runs` (smoke tests excluded) into
`MyDrive/Colab Notebooks/agricl-net/report/`:

* `report.md`: all results as tables, ready to copy into the documentation;
* `key_claim.png`: seen vs. unseen diseases (the project's main claim);
* `continual_learning.png`: accuracy as new diseases are added, per method;
* `edge.png`: size and speed of the edge models, FP32 vs. INT8.

Steps that have not been run yet are marked "Not run yet". No GPU needed: a CPU runtime is enough.

## Storage and project code from GitHub

Works on **Colab** and **Kaggle**:
* Colab: everything is kept in Google Drive, `MyDrive/Colab Notebooks/agricl-net`.
* Kaggle: turn on *Settings > Accelerator > GPU T4* and *Settings > Internet*. Results go to
  `/kaggle/working/agricl-net` and are kept when you *Save Version*. Downloads go to `/tmp` (fetched again each
  session). To use an earlier notebook's results (or resume a run), attach that notebook's output with
  *Add Input > Notebook Output*: it is copied in automatically below.

In [ ]:
import os, sys, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK = '/tmp'                               # scratch space: code, unzipped data
    DRIVE = '/kaggle/working/agricl-net'        # results, saved with each notebook version
    CACHE = '/tmp/agricl-cache'                 # dataset zip and SAM2 checkpoint
    for src in glob.glob('/kaggle/input/**/agricl-net/runs/*', recursive=True):
        dst = f'{DRIVE}/runs/{os.path.basename(src)}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print('copied earlier results:', os.path.basename(src))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    DRIVE = '/content/drive/MyDrive/Colab Notebooks/agricl-net'   # dataset zip, SAM2 checkpoint and all results
    CACHE = f'{DRIVE}/cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

CODE = f'{WORK}/AGRICL-NET'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q https://github.com/Dev02005/AGRICL-NET.git {CODE}
sys.path.insert(0, CODE)
!git -C {CODE} log -1 --format='code version: %h %s'

## Build the report

In [ ]:
from IPython.display import Markdown, display
from agricl.report import build_report, collect

RUNS = f'{DRIVE}/runs'
REPORT = f'{DRIVE}/report'
os.makedirs(REPORT, exist_ok=True)
md_text = build_report(RUNS)
open(f'{REPORT}/report.md', 'w').write(md_text)
res = collect(RUNS)
display(Markdown(md_text))

## Charts

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

def save(fig, name):
    fig.tight_layout(); fig.savefig(f'{REPORT}/{name}', dpi=200); plt.show()

# 1. key claim: seen vs unseen
bars = []
for name, r in res['lesion'].items():
    bars.append(('Lesion Dice', r['test_seen']['mean_dice'], r['unseen']['mean_dice']))
for name, r in res['severity'].items():
    bars.append(('Severity R²', r['seen']['severity_r2'], r['unseen']['severity_r2']))
if bars:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    x = np.arange(len(bars))
    ax.bar(x - 0.18, [b[1] for b in bars], 0.36, label='seen diseases', color='#2e7d32')
    ax.bar(x + 0.18, [b[2] for b in bars], 0.36, label='unseen diseases', color='#f9a825')
    ax.set_xticks(x, [b[0] for b in bars]); ax.set_ylim(0, 1); ax.legend()
    ax.set_title('Lesions and severity on diseases never seen in training')
    save(fig, 'key_claim.png')
else:
    print('key claim: run notebooks 01 and 02 first')

# 2. continual learning curve
for name, methods in res['continual'].items():
    rounds = json.load(open(f'{RUNS}/{name}/rounds.json'))
    seen = np.cumsum([len(r['diseases']) for r in rounds])
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for m, r in methods.items():
        ax.plot(seen[:len(r['step_accuracy'])], r['step_accuracy'], marker='o', label=m)
    joint = next(iter(res['classifier'].values()), {}).get('test_seen', {}).get('top1')
    if joint is not None:
        ax.axhline(joint, color='k', ls='--', label='joint (upper bound)')
    ax.set_xlabel('diseases learned so far'); ax.set_ylabel('accuracy on all of them'); ax.set_ylim(0, 1)
    ax.legend(); ax.set_title('Learning new diseases without forgetting')
    save(fig, 'continual_learning.png')

# 3. edge models
for name, r in res['edge'].items():
    if not r.get('benchmark'):
        continue
    b = r['benchmark']
    labels = [f"{x['model']}\n{x['precision']}" for x in b]
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    ax[0].bar(labels, [float(x['size_mb']) for x in b], color='#2e7d32'); ax[0].set_title('size (MB)')
    ax[1].bar(labels, [float(x['ms_1_thread']) for x in b], color='#1565c0'); ax[1].set_title('ms per image, 1 CPU thread')
    save(fig, 'edge.png')
!ls -la "{REPORT}"

## Next
Copy the tables from `report.md` and the charts into the documentation ("New Document"), in a new
"Results" section, and update the Evaluation Plan with the measured numbers.